In [3]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='file:///c:/Users/MSI/Desktop/MLOps/MLOps_ZoomCamp_Course/02_Experiment_Tracking/mlruns/1', creation_time=1794146536551, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794146536551, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
import pandas as pd

df_train = pd.read_parquet("../Data/green_tripdata_2021-01.parquet")

df_test = pd.read_parquet("../Data/green_tripdata_2021-02.parquet")

In [5]:
df_train["duration"] = (
  df_train["lpep_dropoff_datetime"] - df_train["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_test["duration"] = (
  df_test["lpep_dropoff_datetime"] - df_test["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)]
df_test = df_test[(df_test.duration >= 1) & (df_test.duration <= 60)] 

In [6]:
from sklearn.feature_extraction import DictVectorizer

categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

test_dicts = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(test_dicts)

y_train = df_train["duration"].values
y_test = df_test["duration"].values

In [7]:
import numpy as np

for X in (X_train, X_test):
  X.indices = X.indices.astype(np.int32)
  X.indptr = X.indptr.astype(np.int32)

In [6]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)

print("train", root_mean_squared_error(y_train, model.predict(X_train)))
print("val  ", root_mean_squared_error(y_test, model.predict(X_test)))

train 11.449081717462942
val   12.078673198609463


In [7]:
lasso = Lasso(alpha = 0.1).fit(X_train, y_train)
print("lasso val", root_mean_squared_error(y_test, lasso.predict(X_test)))

lasso val 12.07870203052328


In [ ]:
with mlflow.start_run(run_name="linear_regression_baseline"):
  mlflow.log_param("model", "linear_regression")
  mlflow.log_param("train_data", "green_tripdata_2021-01.parquet")
  mlflow.log_param("valid_data", "green_tripdata_2021-02.parquet")
  mlflow.log_metric("rmse", root_mean_squared_error(y_test, model.predict(X_test)))
  mlflow.log_metric("mae", mean_absolute_error(y_test, model.predict(X_test)))
  mlflow.log_metric("r2", r2_score(y_test, model.predict(X_test)))
  mlflow.log_metric("train_rmse", root_mean_squared_error(y_train, model.predict(X_train)))

In [9]:
for alpha in [0.001, 0.01, 0.1, 1]:
  with mlflow.start_run(run_name=f"lasso_alpha_{alpha}"):
    mlflow.log_param("model", "lasso")
    mlflow.log_param("alpha", alpha)
    lasso = Lasso(alpha=alpha).fit(X_train, y_train)
    mlflow.log_metric("rmse", root_mean_squared_error(y_test, lasso.predict(X_test)))

In [10]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [11]:
train = xgb.DMatrix(X_train, label = y_train)
test = xgb.DMatrix(X_test, label = y_test)

In [12]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
    "seed": 42,
}


In [ ]:
import subprocess

def git_info():
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip() != ""
    return commit, dirty



def objective(params):
  with mlflow.start_run():
    commit, dirty = git_info()
    mlflow.set_tag("model", "xgboost")
    mlflow.set_tag("git_commit", commit)
    mlflow.set_tag("git_dirty", dirty)
    mlflow.log_params(params)
    mlflow.log_param("train_data", "green_tripdata_2021-01.parquet")
    mlflow.log_param("valid_data", "green_tripdata_2021-02.parquet")

    booster = xgb.train(
      params = params,
      dtrain = train,
      num_boost_round = 100,
      evals = [(test, "validation")],
      early_stopping_rounds = 50,
      verbose_eval=False
    )
    y_pred = booster.predict(test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)

  return {
    "loss":rmse,
    "status":STATUS_OK
  }

In [14]:
best_result = fmin(
  fn = objective,
  space = search_space,
  algo=tpe.suggest,
  max_evals = 20,
  trials = Trials(),
)

print(best_result)

100%|██████████| 20/20 [09:51<00:00, 29.55s/trial, best loss: 6.1931215421215615]
{'learning_rate': np.float64(0.3259929087503729), 'max_depth': np.float64(8.0), 'min_child_weight': np.float64(1.7940782636814863), 'reg_alpha': np.float64(0.14101645433874654), 'reg_lambda': np.float64(0.07319224112096427)}


In [ ]:
import pickle
from pathlib import Path

Path("models").mkdir(exist_ok=True)

best_params = {
  "max_depth": int(best_result["max_depth"]),
  "learning_rate": best_result["learning_rate"],
  "reg_alpha": best_result["reg_alpha"],
  "reg_lambda": best_result["reg_lambda"],
  "min_child_weight": best_result["min_child_weight"],
  "objective": "reg:squarederror",
  "seed": 42,
}

with mlflow.start_run(run_name="xgboost_best"):
  commit, dirty = git_info()
  mlflow.set_tag("model", "xgboost")
  mlflow.set_tag("git_commit", commit)
  mlflow.set_tag("git_dirty", dirty)
  mlflow.log_params(best_params)
  mlflow.log_param("train_data", "green_tripdata_2021-01.parquet")
  mlflow.log_param("valid_data", "green_tripdata_2021-02.parquet")

  booster = xgb.train(
    params=best_params,
    dtrain=train,
    num_boost_round=100,
    evals=[(test, "validation")],
    early_stopping_rounds=50,
    verbose_eval=False,
  )

  rmse = root_mean_squared_error(y_test, booster.predict(test))
  mlflow.log_metric("rmse", rmse)

  with open("models/preprocessor.b", "wb") as f:
    pickle.dump(dv, f)
  mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
  mlflow.log_artifact("duration-prediction.ipynb", artifact_path="code")
  mlflow.xgboost.log_model(booster, name="models_mlflow")


2026/11/13 19:19:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [1]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
model = mlflow.xgboost.load_model("models:/nyc-taxi-regressor@champion")  # Replace <RUN_ID> with the actual run ID of the best model

print(model)

In [8]:
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

preds = model.predict(xgb.DMatrix(X_test))
print(root_mean_squared_error(y_test, preds))   # should be about 6.19


6.1931215421215615
